<a href="https://colab.research.google.com/github/ahmedabdelshaheed/flyrank_machinelearning/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahmedabdelshaheed/flyrank_machinelearning/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

one row = one page , in 3/2026

In [1]:
import duckdb, pandas as pd
from getpass import getpass
from huggingface_hub import login
from datasets import load_dataset

TOKEN = getpass("HF token: ")
login(token=TOKEN)

con = duckdb.connect()
con.execute(f"CREATE SECRET hf (TYPE huggingface, TOKEN '{TOKEN}')")
SRC = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet"


START, END = "2026-03-01", "2026-3-31"

df = con.execute(f"""
SELECT content_hash_id,
       client_hash_id,
FROM read_parquet('{SRC}', hive_partitioning=true)
WHERE report_date BETWEEN DATE '{START}' AND DATE '{END}'
GROUP BY client_hash_id, content_hash_id
""").df()

print(df.shape)
df.head()

HF token: ··········


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(331437, 2)


,content_hash_id,client_hash_id
0,content_b7e512995f79d5a6,client_73cda7b4e4f265ea
1,content_05597932fe4da067,client_73cda7b4e4f265ea
2,content_905aa32a0230694e,client_73cda7b4e4f265ea
3,content_05434271b257bb68,client_73cda7b4e4f265ea
4,content_d056587ff7faca0c,client_73cda7b4e4f265ea


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*


**Features**


From `fact_content_daily_performance` :
`gsc_impressions`, `gsc_clicks`,`ga4_users`


**Label**

Percent change in average daily `gsc_impressions` per page between March 2026 and April 2026: `(avg_april - avg_march) / avg_march * 100`.

`down` if below -20%, `up` if above +20%, otherwise `stable`.

**Context**

`client_hash_id`, `content_hash_id`, `report_date`, `gsc_data_start`, `ga4_data_start`, `is_published`, `is_deleted`

**Excluded**

`fact_content_query_90d` table, because its fixed window (2 April to 30 June 2026) does not cover March.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
SRC = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet"

# الفلتر: شهر مارس 2026 + gsc_data_available IS TRUE
df = con.execute(f"""
SELECT *
FROM read_parquet('{SRC}', hive_partitioning=true)
WHERE month = '2026-03'
  AND report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
  AND gsc_data_available IS TRUE
""").df()

# عدد الصفوف بعد الفلتر
print("عدد الصفوف:", len(df))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [ ]:
df.head(20)

,client_hash_id,content_hash_id,n_days,n_days_june
0,client_2094c6eb080311d5,content_15200126d986d645,60,30
1,client_2094c6eb080311d5,content_15caf3515edbfac3,53,30
2,client_2094c6eb080311d5,content_15f758d8bd5341c5,60,30
3,client_2094c6eb080311d5,content_16157539a54a6404,57,30
4,client_2094c6eb080311d5,content_1630bd7c0684d82c,60,30
5,client_2094c6eb080311d5,content_16515c58d44f1f37,60,30
6,client_2094c6eb080311d5,content_167ea9e53c1a8e6e,60,30
7,client_2094c6eb080311d5,content_1696131c8c10f3cd,60,30
8,client_2094c6eb080311d5,content_16a0917ed541a052,57,30
9,client_2094c6eb080311d5,content_16dc713628278346,60,30


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.